Utilizzando il dataset CIFAR-10 (già integrato in Keras), crea un classificatore specializzato solo su 3 classi: Uccelli (2), Gatti (3) e Cani (5).

Filtro Dati: Estrai solo le immagini appartenenti a queste tra classi dal dataset originale.
Pipeline: Implementa una pipeline tf.data che indluda RandomConstrat (0.2) come tecnica di data augmentation.
Deploy: Esporta il modello finale in formato TFLite utilizzando la quantizzazione Float16 (invece di quella di default) per massimizzare la precisione su GPU mobili.

In [10]:
"""
PIPELINE CIFAR-10 - CLASSIFICAZIONE 3 CLASSI
Classi originali CIFAR-10:
2 = bird
3 = cat
5 = dog

Obiettivo:
- filtrare solo bird/cat/dog
- pipeline tf.data
- data augmentation con RandomContrast(0.2)
- classificazione multiclasse a 3 classi
- export TFLite con quantizzazione Float16
"""

import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# ------------------------------------------------------------------
# 0. IMPOSTAZIONI GLOBALI
# ------------------------------------------------------------------
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE = (32, 32)
BATCH_SIZE = 64
EPOCHS = 10

CLASSI_ORIGINALI = [2, 3, 5]
CLASS_NAMES = ["bird", "cat", "dog"]
NUM_CLASSI = 3


In [12]:
"""
PIPELINE CIFAR-10 - CLASSIFICAZIONE 3 CLASSI
Classi originali CIFAR-10:
2 = bird
3 = cat
5 = dog

Obiettivo:
- filtrare solo bird/cat/dog
- pipeline tf.data
- data augmentation con RandomContrast(0.2)
- classificazione multiclasse a 3 classi
- export TFLite con quantizzazione Float16
"""

# ------------------------------------------------------------------
# 0. IMPOSTAZIONI GLOBALI
# ------------------------------------------------------------------
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE = (32, 32)
BATCH_SIZE = 64
EPOCHS = 10

CLASSI_ORIGINALI = [2, 3, 5]
CLASS_NAMES = ["bird", "cat", "dog"]
NUM_CLASSI = 3


In [16]:
# ------------------------------------------------------------------
# 1. CARICAMENTO CIFAR-10
# ------------------------------------------------------------------

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

y_train = y_train.flatten() #porta la forma da (50000,1) a (50000,) per compatibilità con tf.data
#passiamo da [2], [3], [5] a 2, 3, 5
y_test = y_test.flatten()

print("Dataset originale:")
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test: ", x_test.shape)
print("y_test: ", y_test.shape)

Dataset originale:
x_train: (50000, 32, 32, 3)
y_train: (50000,)
x_test:  (10000, 32, 32, 3)
y_test:  (10000,)


In [ ]:

# ------------------------------------------------------------------
# 2. FILTRO CLASSI BIRD, CAT, DOG
# ------------------------------------------------------------------

def filtra_e_rimappa_classi(x, y):
    """
    CIFAR-10 contiene 10 classi.

    Noi vogliamo solo:
    2 = bird; 3 = cat; 5 = dog

    Però il modello finale deve produrre 3 classi:
    0 = bird; 1 = cat; 2 = dog

    Quindi filtriamo (con mask) e poi rimappiamo.
    """

    mask = np.isin(y, CLASSI_ORIGINALI)

    x_filtrato = x[mask]
    y_originale_filtrato = y[mask]

    mapping = {2: 0,3: 1,5: 2}

    y_rimappato = np.array(
        [mapping[int(label)] for label in y_originale_filtrato],
        dtype=np.int32
    )

    return x_filtrato, y_rimappato

#Filtriamo e rimappiamo le classi
x_train, y_train = filtra_e_rimappa_classi(x_train, y_train)
x_test, y_test = filtra_e_rimappa_classi(x_test, y_test)

print("\nDataset filtrato:")
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test: ", x_test.shape)
print("y_test: ", y_test.shape)

#Distribuzione classi training
print("\nDistribuzione classi training:")
for idx, name in enumerate(CLASS_NAMES):
    print(name, ":", np.sum(y_train == idx))



Dataset filtrato:
x_train: (15000, 32, 32, 3)
y_train: (15000,)
x_test:  (3000, 32, 32, 3)
y_test:  (3000,)

Distribuzione classi training:
bird : 5000
cat : 5000
dog : 5000


In [ ]:
# ------------------------------------------------------------------
# 3. PIPELINE tf.data
# ------------------------------------------------------------------

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomContrast(0.2)
], name="data_augmentation")

def prepara_dataset(x, y, training=False):
    """
    Crea una pipeline tf.data.

    Passaggi:
    1. prende array NumPy;
    2. li trasforma in Dataset TensorFlow;
    3. se training=True, mescola i dati;
    4. normalizza i pixel da 0-255 a 0-1;
    5. se training=True, applica RandomContrast;
    6. crea batch;
    7. usa prefetch per migliorare le prestazioni.
    """

    # Devo trasformare due array NumPy in un Dataset TersorFlow. 
    # Ogni elemento del dataset sarà una tuple (img, label)
    # prima di fare tensor_slice ho due array NumPy: x e y che sono tra loro indipendenti.
    # TensorFlow costruisce un dastaset formato da coppie (x,y)
    # ogni elemento del dataset sarù una tupla (img, label)
    ds = tf.data.Dataset.from_tensor_slices((x, y))

    # Se training, mescola i dati per evitare overfitting e migliorare la generalizzazione.
    if training:
        ds = ds.shuffle(buffer_size=len(x),seed=SEED)

    # Normalizzo i pixel da 0-255 e 0-1
    def normalizza(img, label):
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    ds = ds.map(
        normalizza,
        num_parallel_calls=AUTOTUNE
    )

    if training:
        ds = ds.map(
            lambda img, label: (
                data_augmentation(img, training=True),
                label
            ),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE) #lascia a TensorFlow decidere quanti core della CPU utilizzare

    return ds

train_ds = prepara_dataset(x_train, y_train, training=True)
test_ds = prepara_dataset(x_test, y_test, training=False)



Dataset aumentato:
x_train: (15000, 32, 32, 3)
y_train: (15000,)
x_test:  (3000, 32, 32, 3)
y_test:  (3000,)


In [ ]:
# ------------------------------------------------------------------
# 4. MODELLO CNN
# ------------------------------------------------------------------

def build_model():
    """
    Modello CNN semplice ma corretto per CIFAR-10.

    Input:
    immagine 32x32x3

    Output:
    3 neuroni, uno per ciascuna classe:
    bird, cat, dog

    Softmax perché è classificazione multiclasse.
    """

    # Creazione modello sequenziale
    # sequantial vuol dire che i layer sono collegati tra loro in sequenza, uno dopo l'altro.
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(32, 3, padding="same", activation="relu"), # ricercatore di caratteristiche (feature extractor)
        layers.BatchNormalization(),
        layers.MaxPooling2D(), # riduco le dimensioni spaziali dell'immagine

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Flatten(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4), #serve ad evitare overfittin, disattiva casualmente il 40% dei neuroni durante l'addesrarre

        layers.Dense(NUM_CLASSI, activation="softmax") #con softmax per classificazione multiclasse.
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# costruisco il modello
model = build_model()

print("\n=== Architettura modello ===")
model.summary()



=== Architettura modello ===


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 356,803 (1.36 MB)

 Trainable params: 356,355 (1.36 MB)

 Non-trainable params: 448 (1.75 KB)

In [21]:
# ------------------------------------------------------------------
# 5. TRAINING (fitto il modello con early stopping e riduzione del learning rate)
# ------------------------------------------------------------------

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

# fit
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr]
)


Epoch 1/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 29s 95ms/step - accuracy: 0.5369 - loss: 0.9871 - val_accuracy: 0.3437 - val_loss: 3.7252 - learning_rate: 0.0010
Epoch 2/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 90ms/step - accuracy: 0.6259 - loss: 0.8241 - val_accuracy: 0.5947 - val_loss: 0.9350 - learning_rate: 0.0010
Epoch 3/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 26s 107ms/step - accuracy: 0.6677 - loss: 0.7397 - val_accuracy: 0.5913 - val_loss: 0.8905 - learning_rate: 0.0010
Epoch 4/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 89ms/step - accuracy: 0.7107 - loss: 0.6643 - val_accuracy: 0.5783 - val_loss: 1.1106 - learning_rate: 0.0010
Epoch 5/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 83ms/step - accuracy: 0.7447 - loss: 0.5993 - val_accuracy: 0.6937 - val_loss: 0.7289 - learning_rate: 0.0010
Epoch 6/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 84ms/step - accuracy: 0.7757 - loss: 0.5362 - val_accuracy: 0.7287 - val_loss: 0.6401 - learning_rate: 0.0010
Epoch 7/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 84ms/step - accuracy: 0.8026 - 

In [22]:
# ------------------------------------------------------------------
# 6. VALUTAZIONE
# ------------------------------------------------------------------

test_loss, test_acc = model.evaluate(test_ds)

print("\nRisultati finali:")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

def per_class_accuracy(model, ds):
    """
    Calcola accuracy separata per bird, cat, dog.
    Utile perché l'accuracy totale può nascondere errori sbilanciati.
    """
    
    y_true = []
    y_pred = []

    for x_batch, y_batch in ds:
        preds = model.predict(x_batch, verbose=0)
        pred_class = np.argmax(preds, axis=1)

        y_true.extend(y_batch.numpy())
        y_pred.extend(pred_class)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    print("\n--- Accuracy per classe ---")
    for idx, name in enumerate(CLASS_NAMES):
        mask = y_true == idx
        acc = np.mean(y_pred[mask] == y_true[mask])
        print(f"{name:>5}: {acc:.2%}")


per_class_accuracy(model, test_ds)

47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7287 - loss: 0.6401

Risultati finali:
Test loss: 0.6401
Test accuracy: 0.7287

--- Accuracy per classe ---
 bird: 84.50%
  cat: 52.60%
  dog: 81.50%


In [23]:
# ------------------------------------------------------------------
# 7. EXPORT TFLITE FLOAT16
# ------------------------------------------------------------------

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.target_spec.supported_types = [
    tf.float16
]

tflite_float16 = converter.convert()

TFLITE_MODEL_PATH = "cifar10_bird_cat_dog_float16.tflite"

with open(TFLITE_MODEL_PATH, "wb") as f:
    f.write(tflite_float16)

print(f"\n[+] Modello TFLite Float16 salvato: {TFLITE_MODEL_PATH}")

INFO:tensorflow:Assets written to: C:\Users\uberti\AppData\Local\Temp\tmpmg0hig48\assets


INFO:tensorflow:Assets written to: C:\Users\uberti\AppData\Local\Temp\tmpmg0hig48\assets


Saved artifact at 'C:\Users\uberti\AppData\Local\Temp\tmpmg0hig48'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2190132366032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2190132366800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2190130821840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2190130820496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2190132367568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2190130822032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2190130819920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2190130822608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2190130821648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2190130820304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  219